# ZooLake paper split reconstruction and audit

This notebook reconstructs and validates the fixed ZooLake dataset split
provided by the authors in `Data.pickle`.

In [18]:
from pathlib import Path

import pickle
import numpy as np
import pandas as pd
EXPECTED_NUM_CLASSES = 35

In [19]:
KAGGLE_INPUT = Path("/kaggle/input")


data_pickle_candidates = sorted(
    KAGGLE_INPUT.rglob("Data.pickle")
)

classes_file_candidates = sorted(
    KAGGLE_INPUT.rglob("classes_ERIC.npy")
)

dataset_root_candidates = sorted({
    *KAGGLE_INPUT.rglob("zooplankton_0p5x"),
    *KAGGLE_INPUT.rglob("1_zooplankton_0p5x"),
})


print("Data.pickle candidates:")
for path in data_pickle_candidates:
    print(" -", path)


print("\nclasses.npy candidates:")
for path in classes_file_candidates:
    print(" -", path)


print("\nZooLake dataset-root candidates:")
for path in dataset_root_candidates:
    if path.is_dir():
        print(" -", path)

Data.pickle candidates:
 - /kaggle/input/datasets/tsikaripunks/data-pickle-paper-split/Data.pickle

classes.npy candidates:
 - /kaggle/input/datasets/tsikaripunks/classes/classes_ERIC.npy

ZooLake dataset-root candidates:
 - /kaggle/input/datasets/tsikaripunks/zoolake-full/zooplankton_0p5x


In [20]:
def select_single_path(candidates, description):
    """
    Selects a path only when exactly one candidate was found.
    This prevents the notebook from silently choosing the wrong input.
    """
    valid_candidates = [
        path
        for path in candidates
        if path.exists()
    ]

    if len(valid_candidates) == 0:
        raise FileNotFoundError(
            f"No candidate found for: {description}"
        )

    if len(valid_candidates) > 1:
        candidate_list = "\n".join(
            f" - {path}"
            for path in valid_candidates
        )

        raise ValueError(
            f"Multiple candidates found for {description}:\n"
            f"{candidate_list}\n\n"
            "Keep only the correct Kaggle input attached "
            "or select the path manually."
        )

    return valid_candidates[0]


DATA_PICKLE_PATH = select_single_path(
    data_pickle_candidates,
    "Data.pickle",
)

CLASSES_PATH = select_single_path(
    classes_file_candidates,
    "classes_ERIC.npy",
)

DATASET_ROOT = select_single_path(
    [
        path
        for path in dataset_root_candidates
        if path.is_dir()
    ],
    "ZooLake dataset root",
)




assert DATA_PICKLE_PATH.is_file()
assert CLASSES_PATH.is_file()
assert DATASET_ROOT.is_dir()


dataset_folders = sorted(
    path.name
    for path in DATASET_ROOT.iterdir()
    if path.is_dir()
)


print("Data.pickle:")
print(DATA_PICKLE_PATH)

print("\nClasses file:")
print(CLASSES_PATH)

print("\nDataset root:")
print(DATASET_ROOT)

print("\nNumber of folders inside dataset root:")
print(len(dataset_folders))

Data.pickle:
/kaggle/input/datasets/tsikaripunks/data-pickle-paper-split/Data.pickle

Classes file:
/kaggle/input/datasets/tsikaripunks/classes/classes_ERIC.npy

Dataset root:
/kaggle/input/datasets/tsikaripunks/zoolake-full/zooplankton_0p5x

Number of folders inside dataset root:
35


In [21]:
CLASS_NAMES = np.load(CLASSES_PATH).astype(str).tolist()

NUM_CLASSES = len(CLASS_NAMES)

print("Number of classes:", NUM_CLASSES)
print("Class names:")
print(CLASS_NAMES)

assert NUM_CLASSES == EXPECTED_NUM_CLASSES

class_mapping = pd.DataFrame({
    "class_index": range(NUM_CLASSES),
    "label": CLASS_NAMES,
})

display(class_mapping)

Number of classes: 35
Class names:
['aphanizomenon', 'asplanchna', 'asterionella', 'bosmina', 'brachionus', 'ceratium', 'chaoborus', 'conochilus', 'copepod_skins', 'cyclops', 'daphnia', 'daphnia_skins', 'diaphanosoma', 'diatom_chain', 'dinobryon', 'dirt', 'eudiaptomus', 'filament', 'fish', 'fragilaria', 'hydra', 'kellicottia', 'keratella_cochlearis', 'keratella_quadrata', 'leptodora', 'maybe_cyano', 'nauplius', 'paradileptus', 'polyarthra', 'rotifers', 'synchaeta', 'trichocerca', 'unknown', 'unknown_plankton', 'uroglena']


,class_index,label
0,0,aphanizomenon
1,1,asplanchna
2,2,asterionella
3,3,bosmina
4,4,brachionus
5,5,ceratium
6,6,chaoborus
7,7,conochilus
8,8,copepod_skins
9,9,cyclops


In [22]:
with open(DATA_PICKLE_PATH, "rb") as file:
    data = pickle.load(file)


print("Number of stored objects:", len(data))

for position, item in enumerate(data):
    print(
        position,
        type(item).__name__,
        getattr(item, "shape", None),
    )

Number of stored objects: 12
0 ndarray (12560,)
1 ndarray (12560, 128, 128, 3)
2 ndarray (12560, 35)
3 ndarray (2692,)
4 ndarray (2692, 128, 128, 3)
5 ndarray (2692, 35)
6 ndarray (2691,)
7 ndarray (2691, 128, 128, 3)
8 ndarray (2691, 35)
9 ndarray (12560, 111)
10 ndarray (2692, 111)
11 ndarray (2691, 111)


In [23]:
train_filenames = data[0]
train_labels = data[2]

validation_filenames = data[3]
validation_labels = data[5]

test_filenames = data[6]
test_labels = data[8]


print(
    "Training:",
    train_filenames.shape,
    train_labels.shape,
)

print(
    "Validation:",
    validation_filenames.shape,
    validation_labels.shape,
)

print(
    "Test:",
    test_filenames.shape,
    test_labels.shape,
)

Training: (12560,) (12560, 35)
Validation: (2692,) (2692, 35)
Test: (2691,) (2691, 35)


## Δημιουργία του αρχικού manifest

In [24]:
def create_split_dataframe(filenames, one_hot_labels, split_name):
    # Find the class index of each image
    label_indices = np.argmax(one_hot_labels, axis=1)

    # Convert class indices to class names
    labels = [
        CLASS_NAMES[index]
        for index in label_indices
    ]

    return pd.DataFrame({
        "filename": filenames.astype(str),
        "label": labels,
        "split": split_name,
    })


train_df = create_split_dataframe(
    train_filenames,
    train_labels,
    "train",
)

validation_df = create_split_dataframe(
    validation_filenames,
    validation_labels,
    "validation",
)

test_df = create_split_dataframe(
    test_filenames,
    test_labels,
    "test",
)


print("Training samples:", len(train_df))
print("Validation samples:", len(validation_df))
print("Test samples:", len(test_df))

display(train_df.head())
display(validation_df.head())
display(test_df.head())

Training samples: 12560
Validation samples: 2692
Test samples: 2691


,filename,label,split
0,./data/1_zooplankton_0p5x/training/zooplankton...,cyclops,train
1,./data/1_zooplankton_0p5x/training/zooplankton...,cyclops,train
2,./data/1_zooplankton_0p5x/training/zooplankton...,maybe_cyano,train
3,./data/1_zooplankton_0p5x/training/zooplankton...,daphnia,train
4,./data/1_zooplankton_0p5x/training/zooplankton...,cyclops,train


,filename,label,split
0,./data/1_zooplankton_0p5x/training/zooplankton...,filament,validation
1,./data/1_zooplankton_0p5x/training/zooplankton...,daphnia,validation
2,./data/1_zooplankton_0p5x/training/zooplankton...,asplanchna,validation
3,./data/1_zooplankton_0p5x/training/zooplankton...,uroglena,validation
4,./data/1_zooplankton_0p5x/training/zooplankton...,ceratium,validation


,filename,label,split
0,./data/1_zooplankton_0p5x/training/zooplankton...,ceratium,test
1,./data/1_zooplankton_0p5x/training/zooplankton...,eudiaptomus,test
2,./data/1_zooplankton_0p5x/training/zooplankton...,eudiaptomus,test
3,./data/1_zooplankton_0p5x/training/zooplankton...,ceratium,test
4,./data/1_zooplankton_0p5x/training/zooplankton...,maybe_cyano,test


## Αντιστοίχιση με τα αρχεία εικόνων

In [25]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".tif",
    ".tiff",
}


# Find all images inside the ZooLake dataset
all_image_paths = [
    path
    for path in DATASET_ROOT.rglob("*")
    if path.suffix.lower() in IMAGE_EXTENSIONS
]


# Create a lookup using the class and image name
image_lookup = {}

for image_path in all_image_paths:
    folder_name = image_path.relative_to(DATASET_ROOT).parts[0]

    # The physical folder uses a different spelling
    if folder_name == "kellikottia":
        folder_name = "kellicottia"

    key = (folder_name, image_path.name)
    image_lookup[key] = str(image_path)


print("Images found in dataset:", len(all_image_paths))

Images found in dataset: 17943


In [26]:
def add_image_paths(dataframe):
    dataframe = dataframe.copy()

    dataframe["image_name"] = [
        Path(filename).name
        for filename in dataframe["filename"]
    ]

    dataframe["filepath"] = [
        image_lookup.get((label, image_name))
        for label, image_name in zip(
            dataframe["label"],
            dataframe["image_name"],
        )
    ]

    return dataframe


train_df = add_image_paths(train_df)
validation_df = add_image_paths(validation_df)
test_df = add_image_paths(test_df)


print(
    "Missing training images:",
    train_df["filepath"].isna().sum(),
)

print(
    "Missing validation images:",
    validation_df["filepath"].isna().sum(),
)

print(
    "Missing test images:",
    test_df["filepath"].isna().sum(),
)
assert train_df["filepath"].notna().all()
assert validation_df["filepath"].notna().all()
assert test_df["filepath"].notna().all()

display(train_df.head())

Missing training images: 0
Missing validation images: 0
Missing test images: 0


,filename,label,split,image_name,filepath
0,./data/1_zooplankton_0p5x/training/zooplankton...,cyclops,train,SPC-EAWAG-0P5X-1531357072113604-3071035817378-...,/kaggle/input/datasets/tsikaripunks/zoolake-fu...
1,./data/1_zooplankton_0p5x/training/zooplankton...,cyclops,train,SPC-EAWAG-0P5X-1541206751828599-6889252211507-...,/kaggle/input/datasets/tsikaripunks/zoolake-fu...
2,./data/1_zooplankton_0p5x/training/zooplankton...,maybe_cyano,train,SPC-EAWAG-0P5X-1570496558527228-3678536917289-...,/kaggle/input/datasets/tsikaripunks/zoolake-fu...
3,./data/1_zooplankton_0p5x/training/zooplankton...,daphnia,train,SPC-EAWAG-0P5X-1589436595239499-10182004143170...,/kaggle/input/datasets/tsikaripunks/zoolake-fu...
4,./data/1_zooplankton_0p5x/training/zooplankton...,cyclops,train,SPC-EAWAG-0P5X-1535891187935356-1573766176829-...,/kaggle/input/datasets/tsikaripunks/zoolake-fu...


## Έλεγχος επαναλαμβανόμενων ονομάτων και overlaps

In [27]:
split_dataframes = {
    "Training": train_df,
    "Validation": validation_df,
    "Test": test_df,
}


for split_name, dataframe in split_dataframes.items():
    duplicate_mask = dataframe.duplicated(
        subset=["label", "image_name"],
        keep=False,
    )

    extra_duplicates = dataframe.duplicated(
        subset=["label", "image_name"],
        keep="first",
    ).sum()

    print(
        split_name,
        "extra duplicate entries:",
        extra_duplicates,
    )

    if duplicate_mask.any():
        display(
            dataframe.loc[
                duplicate_mask,
                ["label", "image_name", "filepath"],
            ]
        )

Training extra duplicate entries: 1


,label,image_name,filepath
3458,dinobryon,SPC-EAWAG-0P5X-1528850008503168-564006655548-0...,/kaggle/input/datasets/tsikaripunks/zoolake-fu...
7467,dinobryon,SPC-EAWAG-0P5X-1528850008503168-564006655548-0...,/kaggle/input/datasets/tsikaripunks/zoolake-fu...


Validation extra duplicate entries: 0
Test extra duplicate entries: 0


In [28]:
def get_image_keys(dataframe):
    return set(
        zip(
            dataframe["label"],
            dataframe["image_name"],
        )
    )


train_keys = get_image_keys(train_df)
validation_keys = get_image_keys(validation_df)
test_keys = get_image_keys(test_df)


overlaps = {
    "Training - Validation": train_keys & validation_keys,
    "Training - Test": train_keys & test_keys,
    "Validation - Test": validation_keys & test_keys,
}


for split_pair, overlapping_images in overlaps.items():
    print(
        split_pair,
        "overlap:",
        len(overlapping_images),
    )

    if overlapping_images:
        display(
            pd.DataFrame(
                sorted(overlapping_images),
                columns=["label", "image_name"],
            )
        )

Training - Validation overlap: 1


,label,image_name
0,dinobryon,SPC-EAWAG-0P5X-1559498570246055-6403994484253-...


Training - Test overlap: 0
Validation - Test overlap: 1


,label,image_name
0,rotifers,SPC-EAWAG-0P5X-1589537326874154-10282734293288...


## Εφαρμογή των επαληθευμένων αφαιρέσεων

In [29]:
clean_train_df = train_df.copy()
clean_validation_df = validation_df.copy()
clean_test_df = test_df.copy()


# Create a temporary key for each image
for dataframe in [
    clean_train_df,
    clean_validation_df,
    clean_test_df,
]:
    dataframe["image_key"] = list(
        zip(
            dataframe["label"],
            dataframe["image_name"],
        )
    )


# Remove the duplicate entry inside the training set
clean_train_df = clean_train_df.drop_duplicates(
    subset="image_key",
    keep="first",
)


validation_keys = set(clean_validation_df["image_key"])
test_keys = set(clean_test_df["image_key"])

higher_priority_keys = validation_keys | test_keys


clean_train_df = clean_train_df[
    ~clean_train_df["image_key"].isin(
        higher_priority_keys
    )
]


clean_validation_df = clean_validation_df[
    ~clean_validation_df["image_key"].isin(
        test_keys
    )
]

# Remove the temporary key and reset the indices
for dataframe in [
    clean_train_df,
    clean_validation_df,
    clean_test_df,
]:
    dataframe.drop(columns="image_key", inplace=True)
    dataframe.reset_index(drop=True, inplace=True)

In [30]:
clean_train_keys = get_image_keys(clean_train_df)
clean_validation_keys = get_image_keys(clean_validation_df)
clean_test_keys = get_image_keys(clean_test_df)


print("Clean training samples:", len(clean_train_df))
print("Clean validation samples:", len(clean_validation_df))
print("Clean test samples:", len(clean_test_df))

print(
    "Training - Validation overlap:",
    len(clean_train_keys & clean_validation_keys),
)

print(
    "Training - Test overlap:",
    len(clean_train_keys & clean_test_keys),
)

print(
    "Validation - Test overlap:",
    len(clean_validation_keys & clean_test_keys),
)
assert len(train_df) - len(clean_train_df) == 2
assert len(validation_df) - len(clean_validation_df) == 1
assert len(test_df) - len(clean_test_df) == 0

assert not (clean_train_keys & clean_validation_keys)
assert not (clean_train_keys & clean_test_keys)
assert not (clean_validation_keys & clean_test_keys)

Clean training samples: 12558
Clean validation samples: 2691
Clean test samples: 2691
Training - Validation overlap: 0
Training - Test overlap: 0
Validation - Test overlap: 0


## Αποθήκευση του clean manifest

In [31]:
OUTPUT_DIR = Path("/kaggle/working/zoolake_clean_split")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


clean_manifest_df = pd.concat(
    [
        clean_train_df,
        clean_validation_df,
        clean_test_df,
    ],
    ignore_index=True,
)


# Store portable paths relative to DATASET_ROOT
clean_manifest_df["relative_filepath"] = [
    Path(filepath).relative_to(DATASET_ROOT).as_posix()
    for filepath in clean_manifest_df["filepath"]
]


# Keep only the columns needed in future notebooks
clean_manifest_df = clean_manifest_df[
    [
        "image_name",
        "label",
        "split",
        "relative_filepath",
    ]
]


MANIFEST_PATH = OUTPUT_DIR / "zoolake_clean_split_manifest.csv"

clean_manifest_df.to_csv(
    MANIFEST_PATH,
    index=False,
)


print("Manifest saved to:", MANIFEST_PATH)
print()
print(clean_manifest_df["split"].value_counts())
print()
print("Total samples:", len(clean_manifest_df))

Manifest saved to: /kaggle/working/zoolake_clean_split/zoolake_clean_split_manifest.csv

split
train         12558
validation     2691
test           2691
Name: count, dtype: int64

Total samples: 17940


In [32]:
import json


CLASS_NAMES_PATH = OUTPUT_DIR / "zoolake_class_names.json"

with open(
    CLASS_NAMES_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        list(CLASS_NAMES),
        file,
        indent=2,
        ensure_ascii=False,
    )


print("Class names saved to:", CLASS_NAMES_PATH)
print("Number of classes:", len(CLASS_NAMES))

Class names saved to: /kaggle/working/zoolake_clean_split/zoolake_class_names.json
Number of classes: 35


In [33]:
import hashlib


saved_manifest_df = pd.read_csv(MANIFEST_PATH)

with open(MANIFEST_PATH, "rb") as file:
    manifest_sha256 = hashlib.sha256(file.read()).hexdigest()


print("Saved rows:", len(saved_manifest_df))
print("Missing values:", saved_manifest_df.isna().sum().sum())
print("Duplicate rows:", saved_manifest_df.duplicated().sum())
print("Number of classes:", saved_manifest_df["label"].nunique())
print("Manifest SHA-256:", manifest_sha256)

Saved rows: 17940
Missing values: 0
Duplicate rows: 0
Number of classes: 35
Manifest SHA-256: 7e44324400eab63407721d38927a1bd010c4819539a636d6e3d56e2ad92e6d1f
